# LLDA: scores exatos, múltiplas expressões e correção de viés

Este notebook investiga como transformar **logits da LLM em uma distribuição relativa sobre 54 tópicos**, usando os 2.000 resumos completos do AAPD. O modelo permanece congelado.

Não buscamos apenas respostas válidas: calculamos `log P(expressão + EOS | artigo, prompt)` para **todos os candidatos**, sem poda por beams e sem mascarar ou renormalizar o vocabulário em cada passo.

| Variante | Representação | Correção lexical |
|---|---|---|
| Nome único | Nome canônico de cada classe | Não |
| Nome único corrigido | Mesmo nome canônico | Sim |
| Múltiplas expressões | Três expressões por classe | Não |
| Múltiplas expressões corrigidas | Mesmas três expressões | Sim |

Todas as variantes usam **o mesmo prompt com as 162 expressões**. Isso isola agregação e correção neste ensaio. O notebook anterior de trie usava um prompt com 54 nomes; compará-lo diretamente não isolaria apenas o efeito da poda.

Usamos 400 artigos para escolher a força da correção e 1.600 para avaliação exploratória. Esses 2.000 artigos já foram inspecionados em experimentos anteriores: a divisão atual evita usar os rótulos da avaliação na seleção de lambda, mas **não substitui uma avaliação final em dados novos**.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_loader.py').exists())
EXPERIMENT_DIR = ROOT / 'experimento_classificacao_topicos'
sys.path.insert(0, str(EXPERIMENT_DIR))
from llda_scores_exatos import run, REFERENCES
from verbalizadores_54 import CLASSES, VERBALIZERS, EXPRESSIONS
from llda_trie_54 import require_gpu

pd.set_option('display.max_colwidth', 120)
OUTPUT_DIR = EXPERIMENT_DIR / 'resultados' / 'llda_scores_exatos_54'
CACHE_DIR = None
LOCAL_FILES_ONLY = False
CANDIDATE_BATCH = 24
print(require_gpu())
display(pd.DataFrame({'Classe': CLASSES, 'Expressões': VERBALIZERS}))

{'name': 'NVIDIA GeForce RTX 5080', 'free_bytes': 15727525888, 'total_bytes': 16608133120}


,Classe,Expressões
0,cmp-lg,"[Computation and Language (Legacy Category), Computational Language Research (Legacy Category), Language Computation..."
1,cond-mat.dis-nn,"[Disordered systems and neural networks, Disordered Neural Systems, Spin Glasses and Neural Networks]"
2,cond-mat.stat-mech,"[Statistical mechanics, Statistical Physics, Statistical Mechanical Systems]"
3,cs.AI,"[Artificial intelligence, AI and Automated Reasoning, Intelligent Reasoning Systems]"
4,cs.CC,"[Computational complexity, Computational Complexity Theory, Complexity of Computation]"
5,cs.CE,"[Computational engineering finance and science, Scientific and Engineering Computation, Computational Science and Fi..."
6,cs.CG,"[Computational geometry, Geometric Computing, Geometric Algorithms]"
7,cs.CL,"[Computational linguistics and natural language processing, Natural Language Processing, Computational Linguistics]"
8,cs.CR,"[Cryptography and security, Cryptographic Security, Computer Security and Cryptography]"
9,cs.CV,"[Computer vision and pattern recognition, Visual Recognition, Image Understanding]"


## 1. Score exato: cada token conta uma vez

Para uma expressão tokenizada em $t_1,\ldots,t_m$, o score inclui sua terminação:

$$s(v,x)=\sum_{j=1}^{m+1}\log P(t_j\mid x,t_{<j}),\quad t_{m+1}=\mathrm{EOS}.$$

Usamos *teacher forcing*. O cache do prompt é calculado uma vez por artigo e copiado para os lotes de candidatos. Cada ramo tem seu próprio cache; nenhum candidato recebe tokens de outro. O primeiro token é pontuado pelos logits finais do prompt; os seguintes e EOS são pontuados após consumir os tokens anteriores.

Não dividimos pelo comprimento: a soma preserva a probabilidade da sequência completa. Salvamos os comprimentos para investigar seu efeito. A comparação numérica com teacher forcing sem cache está em `test_scores_exatos.py`.

## 2. Agregar expressões e descontar preferência lexical

Cada classe tem exatamente três expressões manuais e pesos iguais. A agregação usa a média das probabilidades, computada em log-space:

$$S(c,x)=\operatorname{logsumexp}_{v\in V_c}s(v,x)-\log|V_c|.$$

Para a correção, pontuamos as mesmas expressões com três entradas neutras: texto vazio, `N/A` e `Scientific abstract.`. Mantemos o catálogo e as instruções. A referência $b(v)$ é o log da média dessas probabilidades.

$$r(v,x)=s(v,x)-\lambda b(v).$$

Agregamos $r$ no lugar de $s$ na variante corrigida. O resultado é um score baseado em razão de probabilidades; não é mais a probabilidade bruta de gerar a expressão. Escolhemos lambda em `{0, 0.25, 0.5, 0.75, 1}` pelo LRAP dos **400 artigos de validação**, separadamente para nome único e múltiplas expressões.

Entradas neutras são uma aproximação da preferência lexical. O catálogo é inicial e precisa de revisão de domínio: expressões amplas podem abranger mais de uma categoria, e qualificadores não resolvem automaticamente categorias equivalentes.

In [2]:
df, metrics, per_class, distributions = run(
    output_dir=OUTPUT_DIR, cache_dir=CACHE_DIR,
    local_files_only=LOCAL_FILES_ONLY, candidate_batch=CANDIDATE_BATCH,
)
display(metrics.round(4))
print(f'{len(df)} resumos completos pontuados; {len(EXPRESSIONS)} candidatos por artigo')

                         Metodo  lambda   LRAP  Ranking loss  Coverage error  AP micro  Acerto top1  F1 micro top3  F1 macro top3  Jaccard top3
                     nome_unico    0.00 0.2148        0.2736         24.3019    0.0936       0.1219         0.1307         0.1011        0.0832
           nome_unico_corrigido    0.75 0.2684        0.2635         23.8738    0.1362       0.2119         0.1908         0.1772        0.1239
           multiplas_expressoes    0.00 0.2341        0.2656         23.5588    0.1007       0.1362         0.1527         0.1251        0.1013
multiplas_expressoes_corrigidas    1.00 0.3556        0.2131         21.8988    0.2214       0.3738         0.2736         0.2064        0.1817
2000 resumos completos pontuados; 162 candidatos por artigo


,Metodo,lambda,LRAP,Ranking loss,Coverage error,AP micro,Acerto top1,F1 micro top3,F1 macro top3,Jaccard top3
0,nome_unico,0.00,0.2148,0.2736,24.3019,0.0936,0.1219,0.1307,0.1011,0.0832
1,nome_unico_corrigido,0.75,0.2684,0.2635,23.8738,0.1362,0.2119,0.1908,0.1772,0.1239
2,multiplas_expressoes,0.00,0.2341,0.2656,23.5588,0.1007,0.1362,0.1527,0.1251,0.1013
3,multiplas_expressoes_corrigidas,1.00,0.3556,0.2131,21.8988,0.2214,0.3738,0.2736,0.2064,0.1817


## 3. Distribuição relativa não é probabilidade multi-rótulo calibrada

Para visualizar a distribuição sobre tópicos, aplicamos softmax aos 54 scores:

$$q(c\mid x)=\frac{\exp S(c,x)}{\sum_k\exp S(k,x)}.$$

Essa saída soma 1 e representa **evidência relativa entre as classes**. Um artigo pode ter vários rótulos verdadeiros; portanto, $q$ não estima automaticamente a probabilidade independente de cada rótulo. Também não afirmamos calibração. Nas variantes corrigidas, softmax normaliza scores de razão de probabilidades.

Os CSVs de log-probabilidades brutas, scores agregados e distribuições normalizadas são separados para manter essa distinção.

In [3]:
for method, values in distributions.items():
    assert values.shape == (2000,54)
    assert np.isfinite(values).all()
    np.testing.assert_allclose(values.sum(1), 1, atol=1e-10)
    print(method, '| soma média:', values.sum(1).mean())

example = pd.DataFrame({method: values[0] for method, values in distributions.items()}, index=CLASSES)
print('Artigo:', df.iloc[0].doc_id, '| classes reais:', df.iloc[0].classes_reais_aapd)
display(example.sort_values('multiplas_expressoes_corrigidas', ascending=False).head(12))

nome_unico | soma média: 1.0
nome_unico_corrigido | soma média: 1.0
multiplas_expressoes | soma média: 1.0
multiplas_expressoes_corrigidas | soma média: 1.0
Artigo: 51 | classes reais: math.CO, cs.DM


,nome_unico,nome_unico_corrigido,multiplas_expressoes,multiplas_expressoes_corrigidas
cs.MA,0.000041,0.006133,0.000071,0.286552
cond-mat.dis-nn,0.689431,0.349458,0.716030,0.141080
math.ST,0.003451,0.112242,0.002201,0.072240
cmp-lg,0.114238,0.242104,0.067870,0.068432
cs.GT,0.062297,0.019567,0.044855,0.060889
cs.CC,0.063597,0.101253,0.058377,0.049170
cs.SI,0.000015,0.001061,0.000217,0.039352
math.CO,0.000152,0.019928,0.000092,0.031069
cs.CR,0.001440,0.001219,0.002514,0.028730
cs.PF,0.000007,0.008458,0.000021,0.025334


## 4. Escolha da correção e avaliação exploratória

LRAP é o critério principal de ranking por documento. Ranking loss e coverage error devem ser menores. AP micro compara scores globalmente e é sensível à escala entre documentos. F1 micro/macro e Jaccard usam **top-3 fixo**, igual para as quatro variantes, sem consultar a quantidade real de rótulos de cada artigo.

As métricas abaixo usam apenas os 1.600 artigos de avaliação. O suporte das 54 classes é exportado, inclusive quando uma classe tem poucos exemplos em um split.

In [4]:
search = pd.read_csv(OUTPUT_DIR / 'busca_lambda_validacao.csv')
support = pd.read_csv(OUTPUT_DIR / 'suporte.csv')
display(search)
display(support)
px.line(search, x='lambda', y='LRAP validacao', color='multiplas_expressoes', markers=True,
        title='Força da correção escolhida apenas na validação').show()
selected = ['LRAP','Ranking loss','Acerto top1','F1 micro top3','F1 macro top3']
plot_data = metrics[['Metodo',*selected]].melt(id_vars='Metodo', var_name='Métrica', value_name='Valor')
fig = px.bar(plot_data, x='Metodo', y='Valor', color='Metodo', facet_col='Métrica', facet_col_wrap=3,
             title='Scores exatos: comparação em 1.600 artigos de avaliação')
fig.update_yaxes(matches=None)
fig.update_layout(height=650, showlegend=False)
fig.show()

,multiplas_expressoes,lambda,LRAP validacao
0,False,0.00,0.198189
1,False,0.25,0.219486
2,False,0.50,0.242498
3,False,0.75,0.256320
4,False,1.00,0.255808
5,True,0.00,0.223392
6,True,0.25,0.256998
7,True,0.50,0.301245
8,True,0.75,0.336574
9,True,1.00,0.345394


,Classe,Validacao,Avaliacao
0,cmp-lg,8,25
1,cond-mat.dis-nn,3,16
2,cond-mat.stat-mech,8,25
3,cs.AI,53,158
4,cs.CC,22,80
5,cs.CE,12,24
6,cs.CG,5,33
7,cs.CL,20,79
8,cs.CR,15,83
9,cs.CV,15,80


Gráfico interativo: comparação dos métodos

Gráfico interativo: comparação dos métodos

## 5. Inspecionar viés, comprimento e diferenças por classe

Expressões com alta probabilidade na referência podem receber scores altos mesmo sem evidência específica do artigo. A correção testa se descontar essa preferência melhora a classificação. O gráfico de comprimento é um diagnóstico descritivo, não prova causal de viés.

Compare também classes favorecidas e prejudicadas: uma melhora global pode esconder perdas em classes raras.

In [5]:
from scipy.special import logsumexp
with np.load(OUTPUT_DIR / 'checkpoint.npz') as checkpoint:
    reference_logp = checkpoint['references']
    raw_logp = checkpoint['logp']
prior = logsumexp(reference_logp, axis=0) - np.log(len(REFERENCES))
catalog = pd.read_csv(OUTPUT_DIR / 'verbalizadores.csv')
catalog['Log probabilidade neutra'] = prior
catalog['Log probabilidade média nos artigos'] = raw_logp.mean(0)
display(catalog.sort_values('Log probabilidade neutra', ascending=False).head(20))
px.scatter(catalog, x='Tokens incluindo EOS', y='Log probabilidade neutra',
           hover_name='Expressao', color='Variante', title='Comprimento e preferência nas entradas neutras').show()
class_table = per_class.pivot(index='Classe', columns='Metodo', values='F1')
class_table['Ganho multi corrigido vs único'] = class_table['multiplas_expressoes_corrigidas'] - class_table['nome_unico']
display(class_table.sort_values('Ganho multi corrigido vs único', ascending=False))
px.bar(per_class, x='Classe', y='F1', color='Metodo', barmode='group', hover_data=['Suporte avaliacao'],
       title='F1 por classe na avaliação exploratória', height=550).show()

,Classe,Variante,Expressao,Tokens incluindo EOS,Log probabilidade neutra,Log probabilidade média nos artigos
23,cs.CL,2,Computational Linguistics,5,-2.682584,-5.986510
7,cond-mat.stat-mech,1,Statistical Physics,4,-3.179953,-6.453455
17,cs.CE,2,Computational Science and Finance,6,-3.500905,-7.454441
16,cs.CE,1,Scientific and Engineering Computation,7,-3.758612,-9.185552
22,cs.CL,1,Natural Language Processing,4,-3.909635,-5.853638
63,cs.LG,0,Machine learning,3,-3.971959,-5.254262
51,cs.GT,0,Computer science and game theory,6,-4.371165,-4.465305
3,cond-mat.dis-nn,0,Disordered systems and neural networks,7,-5.009273,-3.525933
142,q-bio.NC,1,Computational Neuroscience,4,-5.195134,-9.897750
10,cs.AI,1,AI and Automated Reasoning,6,-5.253744,-7.419003


Gráfico interativo: comparação dos métodos

Metodo,multiplas_expressoes,multiplas_expressoes_corrigidas,nome_unico,nome_unico_corrigido,Ganho multi corrigido vs único
Classe,,,,,
physics.soc-ph,0.100000,0.655367,0.000000,0.000000,0.655367
quant-ph,0.043478,0.619718,0.000000,0.042553,0.619718
math.IT,0.007273,0.565380,0.035842,0.423398,0.529538
cs.CG,0.190476,0.433333,0.055556,0.354839,0.377778
math.LO,0.160000,0.375000,0.000000,0.380952,0.375000
cs.DC,0.032258,0.319328,0.000000,0.053333,0.319328
cs.DB,0.217391,0.338983,0.046512,0.272727,0.292471
cs.CR,0.452055,0.532468,0.241135,0.107527,0.291333
math.OC,0.102041,0.400000,0.114286,0.267782,0.285714


Gráfico interativo: comparação dos métodos

## 6. Conclusão calculada e próximos passos

Escolher o melhor LRAP observado serve para orientar a próxima rodada, não para provar superioridade. Revisões dos verbalizadores, lambda e templates devem ser tratadas como desenvolvimento. A contribuição candidata está na combinação e avaliação dos componentes; pontuar nomes, usar verbalizadores e corrigir viés já têm antecedentes na literatura.

Próximos testes: estabilidade diante de ordem/templates, comparação com modelo maior, calibração de pertinência independente por classe e avaliação final em documentos ainda não inspecionados.

In [6]:
best = metrics.loc[metrics.LRAP.idxmax()]
print(f"Maior LRAP exploratório: {best['Metodo']} ({best['LRAP']:.4f}).")
print('Lambdas selecionados:')
display(metrics[['Metodo','lambda']])
print('Scores completos e distribuições relativas salvos em:', OUTPUT_DIR)
display(pd.DataFrame({'Arquivo': sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())}))

Maior LRAP exploratório: multiplas_expressoes_corrigidas (0.3556).
Lambdas selecionados:
Scores completos e distribuições relativas salvos em: /home/jesus/Documentos/LLDA/experimento_classificacao_topicos/resultados/llda_scores_exatos_54


,Metodo,lambda
0,nome_unico,0.00
1,nome_unico_corrigido,0.75
2,multiplas_expressoes,0.00
3,multiplas_expressoes_corrigidas,1.00


,Arquivo
0,busca_lambda_validacao.csv
1,checkpoint.npz
2,configuracao.json
3,distribuicao_relativa_multiplas_expressoes.csv
4,distribuicao_relativa_multiplas_expressoes_corrigidas.csv
5,distribuicao_relativa_nome_unico.csv
6,distribuicao_relativa_nome_unico_corrigido.csv
7,documentos.csv
8,execucao_gpu.json
9,logp_brutos.csv


Referências relacionadas: [verbalizadores em classificação de tópicos](https://aclanthology.org/2024.lrec-main.527/), [correção de vieses de rótulos](https://aclanthology.org/2023.acl-long.783/). Este notebook é uma exploração metodológica, não uma alegação de novidade estabelecida.

## Resultado desta execução

Qwen2.5-0.5B-Instruct na RTX 5080: **324.000 scores de candidatos** nos 2.000 artigos, além das três referências neutras. Inferência e checkpoint: **259,86 segundos**; pico de memória CUDA alocada: **1,71 GiB**.

| Variante | Lambda escolhido na validação | LRAP na avaliação | Acerto top-1 |
|---|---:|---:|---:|
| Nome único | 0 | 0,2148 | 12,19% |
| Nome único corrigido | 0,75 | 0,2684 | 21,19% |
| Múltiplas expressões | 0 | 0,2341 | 13,62% |
| Múltiplas expressões corrigidas | 1 | 0,3556 | 37,38% |

Nesta avaliação exploratória, a combinação apresentou a maior recuperação dos rótulos. O resultado apoia investigar a correção da preferência lexical, mas não estabelece calibração nem superioridade em dados novos. Há quatro distribuições relativas por artigo, com 54 componentes e soma 1, e os scores anteriores à normalização foram preservados.
